#LOAD_Preparation_Data

*Propósito:* Construir la tabla maestra de features (portafolio + variables) aplicando lógica de negocio: Dueños, materialidad, variables AG, antiguedad SUNAT,etc. Escritura particionada por codmes.

In [0]:
%run ../config/config

In [0]:
import sys
import time
import os

sys.path.insert(0, str(PROJECT_PATH))
from utils.logger import MLOpsLogger
from utils.month_delta import month_delta
from utils.data_preparation.universo_implementacion import UniversoImplementacion

from pyspark.sql import SparkSession
import mlflow
import mlflow.tracking._model_registry.utils
mlflow.tracking._model_registry.utils._get_registry_uri_from_spark_session = lambda: "databricks"

In [0]:
logger = MLOpsLogger(
    name='03_load_preparation_data',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO",
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': '03_load_preparation_data'
    }
)

In [0]:
mlflow.set_registry_uri("databricks-uc")
logger.info(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")

In [0]:
RUN_HISTORICAL = FIRST_RUN
if FIRST_RUN and MESES_HISTORICOS_CALIDAD > 0:
    meses_historicos_lista = []
    mes_tmp = MES_VTA
    for i in range(MESES_HISTORICOS_CALIDAD):
        mes_tmp = int(month_delta(str(mes_tmp), -1))
        meses_historicos_lista.append(mes_tmp)
    meses_historicos_lista.reverse()
    meses_a_procesar = meses_historicos_lista + [MES_VTA]
    logger.info(f"MODO HISTÓRICO: {len(meses_historicos_lista)} meses: {meses_historicos_lista}")
else:
    meses_a_procesar = [MES_VTA]
    logger.info(f"Procesando únicamente el mes actual: {MES_VTA}")

spark = SparkSession.builder.getOrCreate()
resultados[]

In [0]:
for idx_mes, mes_actual_proceso in enumerate(meses_a_procesar, 1):
    logger.info("")
    logger.info("=" * 80)
    logger.info(f"PROCESANDO MES {idx_mes}/{len(meses_a_procesar)}: {mes_actual_proceso}")
    logger.info("=" * 80)

    try:
        logger.log_stage_start('load_preparation_data', mes_vta=mes_actual_proceso, environment=ENV)
        start_time = time.time()

        # Instanciar la clase con los parámetros del config
        universo_imp = UniversoImplementacion(
            spark=spark,
            codmes=int(mes_actual_proceso),
            src_catalog="catalog_lhcl_prod.bcp",  # Fuente fija, podría parametrizarse
            sink_catalog=CATALOG_NAME,
            sink_schema=SCHEMA_NAME,
            sink_table=BASE_NAME_TABLE_MASTER
        )

        # Construir la tabla maestra
        universo_imp.execute()

        # Contar registros escritos para este mes
        full_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{BASE_NAME_TABLE_MASTER}"
        count_final = spark.table(full_table_name).filter(F.col("codmes") == mes_actual_proceso).count()
        logger.info(f"Registros escritos en {full_table_name} para codmes={mes_actual_proceso}: {count_final:,}")

        total_duration = time.time() - start_time
        logger.log_stage_end('load_preparation_data', duration=total_duration, records_extracted=count_final)
        resultados.append({
            'mes': mes_actual_proceso,
            'registros': count_final,
            'exitoso': True,
            'duracion': total_duration
        })

    except Exception as e:
        logger.log_exception(
            operation='load_preparation_data',
            exception=e,
            should_raise=False,
            mes_vta=mes_actual_proceso,
            environment=ENV
        )
        resultados.append({
            'mes': mes_actual_proceso,
            'registros': 0,
            'exitoso': False,
            'error': str(e)
        })
        continue  # Continuar con el siguiente mes aunque uno falle

In [0]:
logger.info("\n" + "=" * 80)
logger.info("RESUMEN FINAL")
logger.info("=" * 80)
exitosos = sum(1 for r in resultados if r['exitoso'])
logger.info(f"Meses: {len(resultados)} | OK: {exitosos} | Error: {len(resultados) - exitosos}")

for r in resultados:
    if r['exitoso']:
        logger.info(f"  ✓ {r['mes']}: {r['registros']:,} registros en {r['duracion']:.2f}s")
    else:
        logger.error(f"  X {r['mes']}: {r.get('error', '?')}")

# Si algún mes falló, el pipeline puede fallar o continuar según requerimiento
if exitosos < len(resultados):
    logger.warning("Uno o más meses fallaron, Revise los logs.")
    # Opcional: raise Exception("Error en procesamiento histórico") si se desea fallar el job

if 'logger' in locals():
    logger.info(f"Finalizando notebook: {logger.name}")
    logger.close()

In [0]:
# ==========================================================
# VALIDACIÓN VISUAL DE LA TABLA GENERADA
# ==========================================================
logger.info("\n" + "=" * 80)
logger.info("VALIDACIÓN DE LA TABLA GENERADA")
logger.info("=" * 80)

df_validate = spark.table(OUTPUT_TABLE)

# 1. Número de columnas
num_cols = len(df_validate.columns)
logger.info(f"📊 Número total de columnas en la tabla: {num_cols}")

# 2. Lista de columnas esperadas (FEATURE_NAMES)
columnas_tabla = set(df_validate.columns)
columnas_esperadas = set(FEATURE_NAMES)
faltantes = columnas_esperadas - columnas_tabla
if faltantes:
    logger.warning(f"⚠️ Faltan {len(faltantes)} variables esperadas: {list(faltantes)}")
else:
    logger.info(f"✅ Todas las {len(FEATURE_NAMES)} variables de FEATURE_NAMES están presentes")

# 3. Mostrar una muestra de los datos (5 filas)
logger.info("👀 Muestra de los primeros 5 registros:")
display(df_validate.limit(5))

# 4. Conteo de registros por partición (codmes)
logger.info("📦 Distribución de registros por partición (codmes):")
display(df_validate.groupBy("codmes").count().orderBy("codmes"))

# 5. Listado de columnas de la tabla
df_validate = spark.table(OUTPUT_TABLE)
print("Número de columnas:", len(df_validate.columns))
print("Lista de columnas:")
for col in sorted(df_validate.columns):
    print(f"  - {col}")

# 6. Guardar información en task values para los siguientes notebooks
dbutils.jobs.taskValues.set(key="features_table", value=OUTPUT_TABLE)
dbutils.jobs.taskValues.set(key="features_table_columns", value=num_cols)
dbutils.jobs.taskValues.set(key="features_total_records", value=df_validate.count())

logger.info("✅ Validación completada")